1. cargar el universo y definir el filtro

In [1]:
import pandas as pd 

colegios = pd.read_json('../../web-app/public/data/colegios_universo.json')

def pasa_filtro_copago(rango_colegio, techo_familia):
    if pd.isna(rango_colegio):
        return True #sin información, se considera que pasa el filtro
    return rango_colegio <= techo_familia

In [2]:
colegios['pasa_copago'] = colegios['pago_mensual_rango'].apply(lambda x: pasa_filtro_copago(x, techo_familia=2)) #ejemplo de techo de familia = 2 (rango )
colegios['pasa_copago'].value_counts()

pasa_copago
True     51
False     6
Name: count, dtype: int64

In [3]:
def pasa_filtro_nivel(ofrece_basica, ofrece_media, nivel_familia):
    if not ofrece_basica and not ofrece_media:
        return True  # educación especial pura -> bypass (decisión de hoy)
    if nivel_familia == 'basica':
        return ofrece_basica
    if nivel_familia == 'media':
        return ofrece_media
    if nivel_familia == 'basica_y_media':
        return ofrece_basica and ofrece_media

In [4]:
colegios['pasa_nivel'] = colegios.apply(
    lambda row: pasa_filtro_nivel(row['ofrece_basica'], row['ofrece_media'], 'media'),
    axis=1
)
colegios['pasa_nivel'].value_counts()

pasa_nivel
False    30
True     27
Name: count, dtype: int64

In [5]:
from math import radians, sin, cos, sqrt, atan2

def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2*atan2(sqrt(a), sqrt(1-a))

def pasa_filtro_distancia(lat_colegio, lon_colegio, lat_sector, lon_sector, tipo, radio_km):
    if tipo == 'flexible':
        return True  # blando -> no descalifica, se pondera en Capa 2
    return distancia_km(lat_colegio, lon_colegio, lat_sector, lon_sector) <= radio_km

In [6]:
lat_tc, lon_tc = -33.43913, -70.7411

colegios['pasa_distancia'] = colegios.apply(
    lambda row: pasa_filtro_distancia(row['LATITUD'], row['LONGITUD'], lat_tc, lon_tc, tipo='duro', radio_km=2),
    axis=1
)
colegios['pasa_distancia'].value_counts()

pasa_distancia
False    33
True     24
Name: count, dtype: int64

In [7]:
def pasa_filtros_duros(colegio, prefs):
    if not pasa_filtro_copago(colegio['pago_mensual_rango'], prefs['techo_copago']):
        return False
    if not pasa_filtro_nivel(colegio['ofrece_basica'], colegio['ofrece_media'], prefs['nivel']):
        return False
    if not pasa_filtro_distancia(colegio['LATITUD'], colegio['LONGITUD'], prefs['lat_sector'], prefs['lon_sector'], prefs['tipo_distancia'], prefs.get('radio_km')):
        return False
    return True

In [8]:
prefs_carolina = {
    'techo_copago': 0,
    'nivel': 'media',
    'lat_sector': lat_tc, 'lon_sector': lon_tc,
    'tipo_distancia': 'duro', 'radio_km': 2
}

colegios['pasa_capa1'] = colegios.apply(lambda row: pasa_filtros_duros(row, prefs_carolina), axis=1)
colegios['pasa_capa1'].value_counts()

pasa_capa1
False    52
True      5
Name: count, dtype: int64

In [9]:
tasa_24 = colegios['conteo_denuncias_24'].fillna(0) / colegios['MAT_TOTAL'] * 100
tasa_25 = colegios['conteo_denuncias_25'].fillna(0) / colegios['MAT_TOTAL'] * 100
colegios['tasa_denuncias_promedio'] = (tasa_24 + tasa_25) / 2

In [10]:
simce_cols = [c for c in colegios.columns if c.startswith('difgru_')]
idps_cols = [c for c in colegios.columns if '_difgru_' in c]

colegios['simce_dif_promedio'] = colegios[simce_cols].mean(axis=1)

In [11]:
def normalizar(serie, invertir=False):
    minimo, maximo = serie.min(), serie.max()
    normalizado = (serie - minimo) / (maximo - minimo)
    return 1 - normalizado if invertir else normalizado

In [12]:
lat_tc, lon_tc = -33.43913, -70.7411
colegios['dist'] = colegios.apply(lambda row: distancia_km(row['LATITUD'], row['LONGITUD'], lat_tc, lon_tc), axis=1)

In [13]:
colegios['score_academico'] = normalizar(colegios['simce_dif_promedio'])

colegios['score_seguridad'] = normalizar(colegios['tasa_denuncias_promedio'], invertir=True)
colegios['score_distancia'] = normalizar(colegios['dist'], invertir=True)

In [14]:
colegios['score_academico'] = colegios['score_academico'].fillna(0.5)


In [15]:
def promedio_dimension(prefijo):
    cols = [c for c in colegios.columns if c.startswith(prefijo + '_difgru_')]
    return colegios[cols].mean(axis=1)

colegios['autoestima_dif_promedio'] = promedio_dimension('autoestima')
colegios['habitos_dif_promedio'] = promedio_dimension('habitos')
colegios['participacion_dif_promedio'] = promedio_dimension('participacion')
colegios['clima_dif_promedio'] = promedio_dimension('clima')

colegios['score_autoestima'] = normalizar(colegios['autoestima_dif_promedio']).fillna(0.5)
colegios['score_habitos'] = normalizar(colegios['habitos_dif_promedio']).fillna(0.5)
colegios['score_participacion'] = normalizar(colegios['participacion_dif_promedio']).fillna(0.5)
colegios['score_clima'] = normalizar(colegios['clima_dif_promedio']).fillna(0.5)

# "Seguridad y clima escolar" del quiz combina clima + denuncias (ya tienes score_seguridad)
colegios['score_convivencia'] = (colegios['score_clima'] + colegios['score_seguridad']) / 2

In [16]:
colegios[['score_academico','score_autoestima','score_habitos','score_participacion','score_convivencia','score_seguridad','score_distancia']].describe()

,score_academico,score_autoestima,score_habitos,score_participacion,score_convivencia,score_seguridad,score_distancia
count,57.000000,57.000000,57.000000,57.000000,57.000000,57.000000,57.000000
mean,0.465993,0.517962,0.559800,0.533698,0.708175,0.866643,0.776228
std,0.230273,0.182900,0.216422,0.200596,0.163213,0.172881,0.213134
min,0.000000,0.000000,0.000000,0.000000,0.135755,0.000000,0.000000
25%,0.305128,0.404762,0.494624,0.475248,0.659399,0.835841,0.762408
50%,0.500000,0.500000,0.516129,0.500000,0.721672,0.912313,0.815943
75%,0.530769,0.642857,0.709677,0.673267,0.780647,0.945909,0.879847
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [17]:
pesos_base = {
    'academico':     {'academico':0.50, 'autoestima':0.05, 'habitos':0.05, 'convivencia':0.05, 'participacion':0.05, 'distancia':0.30},
    'autoestima':    {'academico':0.05, 'autoestima':0.50, 'habitos':0.05, 'convivencia':0.05, 'participacion':0.05, 'distancia':0.30},
    'habitos':       {'academico':0.05, 'autoestima':0.05, 'habitos':0.50, 'convivencia':0.05, 'participacion':0.05, 'distancia':0.30},
    'convivencia':   {'academico':0.05, 'autoestima':0.05, 'habitos':0.05, 'convivencia':0.50, 'participacion':0.05, 'distancia':0.30},
    'participacion': {'academico':0.05, 'autoestima':0.05, 'habitos':0.05, 'convivencia':0.05, 'participacion':0.50, 'distancia':0.30},
    'todo_por_igual':{'academico':0.14, 'autoestima':0.14, 'habitos':0.14, 'convivencia':0.14, 'participacion':0.14, 'distancia':0.30},
}

def calcular_pesos_finales(perfil, tipo_distancia):
    pesos = pesos_base[perfil].copy()
    if tipo_distancia == 'duro':
        pesos.pop('distancia')
        total = sum(pesos.values())
        pesos = {k: v / total for k, v in pesos.items()}
    return pesos

def calcular_score(colegio, perfil, tipo_distancia, quiere_inclusion, peso_bono_inclusion=0.05):
    pesos = calcular_pesos_finales(perfil, tipo_distancia)
    pesos = {k: v * (1 - peso_bono_inclusion) for k, v in pesos.items()}
    score = sum(colegio[f'score_{dim}'] * peso for dim, peso in pesos.items())
    if quiere_inclusion and (colegio['ofrece_educacion_especial'] or colegio['CONVENIO_PIE']):
        score += peso_bono_inclusion
    return score

In [18]:
colegios['pasa_capa1'] = colegios.apply(lambda row: pasa_filtros_duros(row, prefs_carolina), axis=1)

elegibles = colegios[colegios['pasa_capa1']].copy()
elegibles['score_final'] = elegibles.apply(
    lambda row: calcular_score(row, perfil='convivencia', tipo_distancia='duro', quiere_inclusion=False),
    axis=1
)
elegibles[['NOM_RBD','score_final']].sort_values('score_final', ascending=False).head(5)

,NOM_RBD,score_final
0,LICEO CENTRO EXPERIMENTAL PUDAHUEL CAREN,0.702098
19,COLEGIO BICENTENARIO MADRE ANA EUGENIA,0.668311
40,ESCUELA ESPECIAL N°1761 MI PEQUEÑO COLIBRI PUD...,0.644643
42,ESCUELA ESPECIAL PART. ACONCAGUA DE PUDAHEL N,0.644643
35,ESCUELA ESPECIAL PART.JESUS NAZARENO,0.606927


In [19]:
prefs_b = {'techo_copago': 5, 'nivel': 'basica', 'lat_sector': lat_tc, 'lon_sector': lon_tc, 'tipo_distancia': 'flexible', 'radio_km': None}

colegios['pasa_b'] = colegios.apply(lambda row: pasa_filtros_duros(row, prefs_b), axis=1)
elegibles_b = colegios[colegios['pasa_b']].copy()
elegibles_b['score_final'] = elegibles_b.apply(
    lambda row: calcular_score(row, perfil='academico', tipo_distancia='flexible', quiere_inclusion=False),
    axis=1
)
elegibles_b[['NOM_RBD','score_final']].sort_values('score_final', ascending=False).head(5)

,NOM_RBD,score_final
47,COLEGIO TERRA AUSTRALIS,0.856432
1,ESCUELA TENIENTE HERNAN MERINO CORREA,0.803234
38,COLEGIO POLIV. GERONIMO DE ALDERETE,0.801807
51,ESCUELA BAS. VILLA SAN IGNACIO,0.759264
26,COLEGIO PARTICULAR LA UNION,0.753868


In [20]:
prefs_c = {'techo_copago': 0, 'nivel': 'basica', 'lat_sector': lat_tc, 'lon_sector': lon_tc, 'tipo_distancia': 'duro', 'radio_km': 3}

colegios['pasa_c'] = colegios.apply(lambda row: pasa_filtros_duros(row, prefs_c), axis=1)
elegibles_c = colegios[colegios['pasa_c']].copy()
elegibles_c['score_final'] = elegibles_c.apply(
    lambda row: calcular_score(row, perfil='todo_por_igual', tipo_distancia='duro', quiere_inclusion=True),
    axis=1
)
elegibles_c[['NOM_RBD','score_final']].sort_values('score_final', ascending=False).head(5)

,NOM_RBD,score_final
26,COLEGIO PARTICULAR LA UNION,0.852355
51,ESCUELA BAS. VILLA SAN IGNACIO,0.835870
1,ESCUELA TENIENTE HERNAN MERINO CORREA,0.830034
31,LICEO BICENTENARIO MONSENOR ENRIQUE ALVEAR,0.813540
18,ESCUELA BASICA N° 912 ELLEN COLLEGE,0.710529
